<a href="https://colab.research.google.com/github/Tazin17/pain-intensity-prediction-from-laser-evoked-eeg-responses/blob/main/V9_EXP8_ExtraTrees_cell_by_cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V9 EXP8 pain-classification pipeline

This notebook keeps the **same Extra Trees configuration used in V8**:

- `n_estimators=150`
- `max_depth=12`
- `min_samples_leaf=3`
- `min_samples_split=5`
- `max_features="sqrt"`
- `class_weight="balanced"`
- `random_state=42`

There is **no Extra Trees hyperparameter search** in this notebook.

Experiments:

1. **V8 strict baseline:** 262 handcrafted features, fixed threshold 0.50.
2. **V9-A:** same 262 features, training-only sigmoid calibration and nested threshold selection.
3. **V9-B:** 262 handcrafted + 52 compact spatial features, same calibration and threshold procedure.
4. **V9-C:** 262 handcrafted + 156 baseline-whitened log-covariance features, same calibration and threshold procedure.
5. Optional within-individual evaluation using the same feature matrices.

The cross-individual outer validation is Leave-One-Subject-Out over 223 participants.

In [ ]:
import os
from google.colab import drive

# Try to unmount if it's already mounted or partially mounted
try:
  drive.flush_and_unmount()
  print('Drive unmounted.')
except ValueError:
  pass # Drive was not mounted, nothing to unmount

# Ensure the mount point is clean
if os.path.isdir('/content/drive'):
  # Remove all contents of the directory, but not the directory itself
  # This avoids issues if the directory is still seen as a mount point
  for item in os.listdir('/content/drive'):
    item_path = os.path.join('/content/drive', item)
    if os.path.isfile(item_path):
      os.remove(item_path)
    elif os.path.isdir(item_path):
      import shutil
      shutil.rmtree(item_path)
  print('Cleared /content/drive directory.')
else:
  # If the directory doesn't exist, create it
  os.makedirs('/content/drive', exist_ok=True)
  print('Created /content/drive directory.')

# Attempt to mount again
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Drive unmounted.
Created /content/drive directory.
Mounted at /content/drive


In [ ]:
# !pip -q install pymatreader mne scipy scikit-learn pandas numpy matplotlib tqdm

In [ ]:
# ============================================================
# CELL 1 — Import libraries
# ============================================================

import os
import re
import json
import time
import warnings
import glob

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
)

print("Imports completed.")
print("NumPy version:", np.__version__)

Imports completed.
NumPy version: 2.0.2


In [ ]:
# ============================================================
# CELL 2 — V8/V9 paths and saved-data loader
# ============================================================

V8_DIR = (
    "/content/drive/MyDrive/"
    "EXP8_CLASSIFICATION_FROM_SCRATCH/"
    "V8_MAIN_PIPELINE_12CH_FULL3S_252"
)

V9_DIR = (
    "/content/drive/MyDrive/"
    "EXP8_CLASSIFICATION_FROM_SCRATCH/"
    "V9_MAIN_PIPELINE_12CH_SPATIAL"
)

V8_FEATURE_DIR = os.path.join(V8_DIR, "handcrafted_features")
V8_ALIGNED_DIR = os.path.join(V8_DIR, "aligned_data")
V8_PREPARED_DIR = os.path.join(V8_DIR, "prepared_data")

V9_FEATURE_DIR = os.path.join(V9_DIR, "features")
V9_CROSS_RESULTS_DIR = os.path.join(V9_DIR, "cross_individual_results")
V9_WITHIN_RESULTS_DIR = os.path.join(V9_DIR, "within_individual_results")

for directory in [
    V9_DIR,
    V9_FEATURE_DIR,
    V9_CROSS_RESULTS_DIR,
    V9_WITHIN_RESULTS_DIR,
]:
    os.makedirs(directory, exist_ok=True)

X_262_PATH = os.path.join(
    V8_FEATURE_DIR,
    "X_v8_handcrafted_262_full0_2000ms.npy"
)

FEATURE_NAMES_262_PATH = os.path.join(
    V8_FEATURE_DIR,
    "v8_handcrafted_feature_names_262.csv"
)

Y_PATH = os.path.join(
    V8_ALIGNED_DIR,
    "y_v8_binary_6275.npy"
)

GROUPS_PATH = os.path.join(
    V8_ALIGNED_DIR,
    "groups_v8_subject_6275.npy"
)

BASELINE_EEG_PATH = os.path.join(
    V8_PREPARED_DIR,
    "X_v8_baseline_raw_12ch_1000samples.npy"
)

RESPONSE_EEG_PATH = os.path.join(
    V8_PREPARED_DIR,
    "X_v8_response_raw_12ch_2000samples.npy"
)

required_paths = [
    X_262_PATH,
    FEATURE_NAMES_262_PATH,
    Y_PATH,
    GROUPS_PATH,
    BASELINE_EEG_PATH,
    RESPONSE_EEG_PATH,
]

missing_paths = [path for path in required_paths if not os.path.exists(path)]

if missing_paths:
    raise FileNotFoundError(
        "Required V8 files were not found:\n\n"
        + "\n".join(missing_paths)
    )

X_v8_262 = np.load(X_262_PATH, mmap_mode="r")
y = np.load(Y_PATH).astype(np.int8, copy=False).ravel()
groups = np.load(GROUPS_PATH, allow_pickle=True).astype(str).ravel()

X_baseline_eeg = np.load(BASELINE_EEG_PATH, mmap_mode="r")
X_response_eeg = np.load(RESPONSE_EEG_PATH, mmap_mode="r")

feature_names_262_df = pd.read_csv(FEATURE_NAMES_262_PATH)

if "feature_name" in feature_names_262_df.columns:
    feature_names_262 = (
        feature_names_262_df["feature_name"].astype(str).tolist()
    )
else:
    feature_names_262 = (
        feature_names_262_df.iloc[:, -1].astype(str).tolist()
    )

CHANNELS = [
    "Fz", "FC1", "FC2", "C3", "C1", "Cz",
    "C2", "C4", "CP1", "CPz", "CP2", "Pz"
]

FS = 1000
N_EPOCHS = 6275
N_SUBJECTS = 223

assert X_v8_262.shape == (6275, 262)
assert X_baseline_eeg.shape == (6275, 12, 1000)
assert X_response_eeg.shape == (6275, 12, 2000)
assert y.shape == (6275,)
assert groups.shape == (6275,)
assert len(feature_names_262) == 262
assert len(np.unique(groups)) == 223
assert np.array_equal(
    np.bincount(y, minlength=2),
    np.array([2725, 3550])
)

print("=" * 72)
print("V9 INPUT DATA READY")
print("=" * 72)
print("V8 handcrafted:", X_v8_262.shape)
print("Baseline EEG:", X_baseline_eeg.shape)
print("Response EEG:", X_response_eeg.shape)
print("Participants:", len(np.unique(groups)))
print("Class counts [low, high]:", np.bincount(y, minlength=2).tolist())


V9 INPUT DATA READY
V8 handcrafted: (6275, 262)
Baseline EEG: (6275, 12, 1000)
Response EEG: (6275, 12, 2000)
Participants: 223
Class counts [low, high]: [2725, 3550]


In [ ]:
# ============================================================
# CELL 3 — Fixed V8 Extra Trees configuration
# ============================================================

def make_v8_extratrees():
    return ExtraTreesClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=3,
        min_samples_split=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )


THRESHOLD_GRID = np.round(
    np.arange(0.40, 0.751, 0.01),
    2
)

# Predeclared Huang benchmark constraints.
TARGET_PRECISION = 0.7308
TARGET_SPECIFICITY = 0.7402
TARGET_RECALL = 0.5414

INNER_GROUP_FOLDS = 5

print("Fixed Extra Trees model:")
print(make_v8_extratrees())
print("Threshold grid:", THRESHOLD_GRID[0], "to", THRESHOLD_GRID[-1])


Fixed Extra Trees model:
ExtraTreesClassifier(class_weight='balanced', max_depth=12, min_samples_leaf=3,
                     min_samples_split=5, n_estimators=150, n_jobs=-1,
                     random_state=42)
Threshold grid: 0.4 to 0.75


In [ ]:
# ============================================================
# CELL 4 — Shared metric, calibration and threshold helpers
# ============================================================

def atomic_write_csv(dataframe, path):
    temporary_path = f"{path}.tmp"
    dataframe.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def atomic_write_json(content, path):
    temporary_path = f"{path}.tmp"
    with open(temporary_path, "w", encoding="utf-8") as file:
        json.dump(content, file, indent=4, default=str)
    os.replace(temporary_path, path)


def safe_name(text):
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(text).strip()
    ).strip("_")


def positive_probability(fitted_model, X):
    classes = np.asarray(fitted_model.classes_)
    locations = np.flatnonzero(classes == 1)

    if len(locations) != 1:
        raise ValueError("Positive class 1 was not found uniquely.")

    probabilities = np.asarray(
        fitted_model.predict_proba(X),
        dtype=float
    )

    return probabilities[:, int(locations[0])].ravel()


def binary_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true, dtype=np.int8).ravel()
    y_pred = np.asarray(y_pred, dtype=np.int8).ravel()

    if y_score is not None:
        y_score = np.asarray(y_score, dtype=float).ravel()

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    has_both_classes = len(np.unique(y_true)) == 2

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": (
            balanced_accuracy_score(y_true, y_pred)
            if has_both_classes
            else np.nan
        ),
        "precision": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "sensitivity": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "specificity": specificity,
        "negative_predictive_value": npv,
        "f1_score": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "roc_auc": (
            roc_auc_score(y_true, y_score)
            if has_both_classes
            and y_score is not None
            and np.isfinite(y_score).all()
            else np.nan
        ),
        "average_precision_pr_auc": (
            average_precision_score(y_true, y_score)
            if has_both_classes
            and y_score is not None
            and np.isfinite(y_score).all()
            else np.nan
        ),
        "matthews_correlation_coefficient": (
            matthews_corrcoef(y_true, y_pred)
        ),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "support_low_pain": int(np.sum(y_true == 0)),
        "support_high_pain": int(np.sum(y_true == 1)),
    }

    return metrics


def fit_sigmoid_calibrator(raw_scores, labels):
    raw_scores = np.asarray(raw_scores, dtype=float).reshape(-1, 1)
    labels = np.asarray(labels, dtype=np.int8).ravel()

    if len(np.unique(labels)) != 2:
        raise ValueError("Calibration labels must contain both classes.")

    calibrator = LogisticRegression(
        solver="lbfgs",
        C=1.0,
        max_iter=1000,
        random_state=42,
    )

    calibrator.fit(raw_scores, labels)
    return calibrator


def apply_sigmoid_calibrator(calibrator, raw_scores):
    raw_scores = np.asarray(raw_scores, dtype=float).reshape(-1, 1)
    return calibrator.predict_proba(raw_scores)[:, 1]


def choose_threshold(
    y_true,
    scores,
    threshold_grid=THRESHOLD_GRID,
    target_precision=TARGET_PRECISION,
    target_specificity=TARGET_SPECIFICITY,
    target_recall=TARGET_RECALL,
):
    rows = []

    for threshold in threshold_grid:
        predictions = (
            np.asarray(scores, dtype=float) >= threshold
        ).astype(np.int8)

        metric = binary_metrics(
            y_true,
            predictions,
            scores
        )

        feasible = (
            metric["precision"] >= target_precision
            and metric["specificity"] >= target_specificity
            and metric["recall"] >= target_recall
        )

        normalized_shortfall = (
            max(
                0.0,
                target_precision - metric["precision"]
            ) / target_precision
            +
            max(
                0.0,
                target_specificity - metric["specificity"]
            ) / target_specificity
            +
            max(
                0.0,
                target_recall - metric["recall"]
            ) / target_recall
        )

        rows.append({
            "threshold": float(threshold),
            "feasible": bool(feasible),
            "constraint_shortfall": float(normalized_shortfall),
            **metric,
        })

    table = pd.DataFrame(rows)
    feasible_table = table[table["feasible"]].copy()

    if len(feasible_table) > 0:
        selected = (
            feasible_table
            .sort_values(
                [
                    "balanced_accuracy",
                    "accuracy",
                    "f1_score",
                    "specificity",
                    "precision",
                ],
                ascending=False,
            )
            .iloc[0]
        )
        selection_reason = "all_constraints_satisfied"
    else:
        selected = (
            table
            .sort_values(
                [
                    "constraint_shortfall",
                    "balanced_accuracy",
                    "accuracy",
                    "f1_score",
                ],
                ascending=[True, False, False, False],
            )
            .iloc[0]
        )
        selection_reason = "minimum_constraint_shortfall"

    return (
        float(selected["threshold"]),
        selected.to_dict(),
        selection_reason,
        table,
    )


def inner_group_oof_scores(
    X_train,
    y_train,
    groups_train,
    estimator,
    n_splits=5,
):
    y_train = np.asarray(y_train, dtype=np.int8).ravel()
    groups_train = np.asarray(groups_train).astype(str).ravel()

    unique_groups = np.unique(groups_train)
    effective_splits = min(int(n_splits), len(unique_groups))

    if effective_splits < 2:
        raise ValueError("At least two training groups are required.")

    splitter = GroupKFold(n_splits=effective_splits)
    oof_scores = np.full(len(y_train), np.nan, dtype=float)

    for inner_train_indices, inner_validation_indices in splitter.split(
        X_train,
        y_train,
        groups_train,
    ):
        inner_model = clone(estimator)

        inner_model.fit(
            X_train[inner_train_indices],
            y_train[inner_train_indices],
        )

        oof_scores[inner_validation_indices] = positive_probability(
            inner_model,
            X_train[inner_validation_indices],
        )

    if not np.isfinite(oof_scores).all():
        raise RuntimeError("Inner OOF probabilities are incomplete.")

    return oof_scores


print("Shared V9 helpers are ready.")


Shared V9 helpers are ready.


In [ ]:
# ============================================================
# CELL 5 — Leakage-free cross-individual LOSO evaluator
# ============================================================

def run_v9_cross_loso(
    X,
    y,
    groups,
    estimator,
    model_name,
    feature_set_name,
    experiment_tag,
    nested_threshold,
    calibrate,
    fixed_threshold=0.50,
    inner_group_folds=INNER_GROUP_FOLDS,
    threshold_grid=THRESHOLD_GRID,
    resume=True,
    overwrite=False,
):
    y = np.asarray(y, dtype=np.int8).ravel()
    groups = np.asarray(groups).astype(str).ravel()

    if X.shape[0] != len(y) or len(y) != len(groups):
        raise ValueError("X, y and groups are not aligned.")

    if len(np.unique(groups)) != 223:
        raise ValueError("Expected 223 participants.")

    if not np.array_equal(
        np.bincount(y, minlength=2),
        np.array([2725, 3550])
    ):
        raise ValueError("Unexpected V9 class counts.")

    experiment_name = (
        f"{safe_name(model_name)}__"
        f"{safe_name(feature_set_name)}__"
        f"{safe_name(experiment_tag)}"
    )

    prediction_path = os.path.join(
        V9_CROSS_RESULTS_DIR,
        experiment_name + "__trial_predictions.csv"
    )

    subject_path = os.path.join(
        V9_CROSS_RESULTS_DIR,
        experiment_name + "__subject_results.csv"
    )

    summary_path = os.path.join(
        V9_CROSS_RESULTS_DIR,
        experiment_name + "__pooled_summary.csv"
    )

    progress_path = os.path.join(
        V9_CROSS_RESULTS_DIR,
        experiment_name + "__progress.json"
    )

    configuration_path = os.path.join(
        V9_CROSS_RESULTS_DIR,
        experiment_name + "__configuration.json"
    )

    output_paths = [
        prediction_path,
        subject_path,
        summary_path,
        progress_path,
        configuration_path,
    ]

    if overwrite:
        for path in output_paths:
            if os.path.exists(path):
                os.remove(path)

    configuration = {
        "experiment_name": experiment_name,
        "validation": "outer Leave-One-Subject-Out",
        "model": repr(estimator),
        "feature_set_name": feature_set_name,
        "n_epochs": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_subjects": int(len(np.unique(groups))),
        "nested_threshold": bool(nested_threshold),
        "calibrate": bool(calibrate),
        "fixed_threshold": float(fixed_threshold),
        "inner_group_folds": int(inner_group_folds),
        "threshold_grid": [float(value) for value in threshold_grid],
        "target_precision": TARGET_PRECISION,
        "target_specificity": TARGET_SPECIFICITY,
        "target_recall": TARGET_RECALL,
        "leakage_control": (
            "All calibration and threshold selection use only "
            "outer-training participants. The outer test participant "
            "is excluded from model fitting, calibration and threshold selection."
        ),
    }

    atomic_write_json(configuration, configuration_path)

    if resume and os.path.exists(prediction_path):
        predictions_df = pd.read_csv(prediction_path)
        predictions_df["test_subject"] = (
            predictions_df["test_subject"].astype(str)
        )
        predictions_df = (
            predictions_df
            .sort_values("original_row_index")
            .drop_duplicates("original_row_index", keep="last")
            .reset_index(drop=True)
        )
        completed_subjects = set(
            predictions_df["test_subject"].unique()
        )
        print(
            "Checkpoint found:",
            len(completed_subjects),
            "/ 223 participants"
        )
    else:
        predictions_df = pd.DataFrame()
        completed_subjects = set()

    all_subjects = np.sort(np.unique(groups))
    wall_start = time.perf_counter()

    for test_subject in tqdm(
        all_subjects,
        desc=f"V9 LOSO | {model_name} | {feature_set_name}",
    ):
        test_subject = str(test_subject)

        if test_subject in completed_subjects:
            continue

        train_indices = np.flatnonzero(groups != test_subject)
        test_indices = np.flatnonzero(groups == test_subject)

        X_train = X[train_indices]
        X_test = X[test_indices]
        y_train = y[train_indices]
        y_test = y[test_indices]
        groups_train = groups[train_indices]

        if np.any(groups_train == test_subject):
            raise RuntimeError("Outer subject leakage detected.")

        if len(np.unique(y_train)) != 2:
            raise RuntimeError("Outer training fold lacks one class.")

        if nested_threshold:
            raw_oof_scores = inner_group_oof_scores(
                X_train=X_train,
                y_train=y_train,
                groups_train=groups_train,
                estimator=estimator,
                n_splits=inner_group_folds,
            )

            if calibrate:
                calibrator = fit_sigmoid_calibrator(
                    raw_oof_scores,
                    y_train,
                )
                inner_scores = apply_sigmoid_calibrator(
                    calibrator,
                    raw_oof_scores,
                )
            else:
                calibrator = None
                inner_scores = raw_oof_scores

            (
                selected_threshold,
                selected_inner_row,
                threshold_reason,
                _,
            ) = choose_threshold(
                y_true=y_train,
                scores=inner_scores,
                threshold_grid=threshold_grid,
            )
        else:
            calibrator = None
            selected_threshold = float(fixed_threshold)
            threshold_reason = "fixed_threshold"
            selected_inner_row = {
                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "specificity": np.nan,
                "f1_score": np.nan,
            }

        final_model = clone(estimator)

        fit_start = time.perf_counter()
        final_model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - fit_start

        raw_test_scores = positive_probability(
            final_model,
            X_test,
        )

        if calibrator is not None:
            final_test_scores = apply_sigmoid_calibrator(
                calibrator,
                raw_test_scores,
            )
            calibration_coefficient = float(
                calibrator.coef_.ravel()[0]
            )
            calibration_intercept = float(
                calibrator.intercept_.ravel()[0]
            )
        else:
            final_test_scores = raw_test_scores
            calibration_coefficient = np.nan
            calibration_intercept = np.nan

        y_pred = (
            final_test_scores >= selected_threshold
        ).astype(np.int8)

        fold_df = pd.DataFrame({
            "experiment": experiment_name,
            "model": model_name,
            "feature_set": feature_set_name,
            "test_subject": test_subject,
            "original_row_index": test_indices,
            "y_true": y_test,
            "y_pred": y_pred,
            "raw_probability": raw_test_scores,
            "y_score": final_test_scores,
            "selected_threshold": selected_threshold,
            "threshold_selection_reason": threshold_reason,
            "calibration_coefficient": calibration_coefficient,
            "calibration_intercept": calibration_intercept,
            "inner_accuracy": selected_inner_row["accuracy"],
            "inner_balanced_accuracy": selected_inner_row[
                "balanced_accuracy"
            ],
            "inner_precision": selected_inner_row["precision"],
            "inner_recall": selected_inner_row["recall"],
            "inner_specificity": selected_inner_row["specificity"],
            "inner_f1_score": selected_inner_row["f1_score"],
            "fit_seconds": fit_seconds,
        })

        predictions_df = pd.concat(
            [predictions_df, fold_df],
            ignore_index=True,
        )

        predictions_df = (
            predictions_df
            .sort_values("original_row_index")
            .drop_duplicates("original_row_index", keep="last")
            .reset_index(drop=True)
        )

        atomic_write_csv(predictions_df, prediction_path)

        completed_subjects.add(test_subject)

        atomic_write_json(
            {
                "completed": False,
                "completed_subjects": len(completed_subjects),
                "total_subjects": 223,
                "completed_prediction_rows": len(predictions_df),
                "total_prediction_rows": len(y),
                "last_completed_subject": test_subject,
            },
            progress_path,
        )

    predictions_df = pd.read_csv(prediction_path)
    predictions_df["test_subject"] = (
        predictions_df["test_subject"].astype(str)
    )
    predictions_df = (
        predictions_df
        .sort_values("original_row_index")
        .drop_duplicates("original_row_index", keep="last")
        .reset_index(drop=True)
    )

    if len(predictions_df) != len(y):
        raise RuntimeError(
            f"Incomplete LOSO predictions: "
            f"{len(predictions_df)} of {len(y)}."
        )

    if not np.array_equal(
        predictions_df["original_row_index"].to_numpy(int),
        np.arange(len(y)),
    ):
        raise RuntimeError("Prediction rows are not fully aligned.")

    if not np.array_equal(
        predictions_df["y_true"].to_numpy(np.int8),
        y,
    ):
        raise RuntimeError("Saved labels are not aligned.")

    subject_rows = []

    for subject_id, subject_df in predictions_df.groupby(
        "test_subject",
        sort=True,
    ):
        metric = binary_metrics(
            subject_df["y_true"],
            subject_df["y_pred"],
            subject_df["y_score"],
        )

        subject_rows.append({
            "subject_id": str(subject_id),
            "n_trials": int(len(subject_df)),
            "selected_threshold": float(
                subject_df["selected_threshold"].iloc[0]
            ),
            **metric,
        })

    subject_results_df = pd.DataFrame(subject_rows)
    atomic_write_csv(subject_results_df, subject_path)

    pooled_metrics = binary_metrics(
        predictions_df["y_true"],
        predictions_df["y_pred"],
        predictions_df["y_score"],
    )

    summary = {
        "experiment": experiment_name,
        "model": model_name,
        "feature_set": feature_set_name,
        "n_epochs": int(len(predictions_df)),
        "n_features": int(X.shape[1]),
        "n_subjects": 223,
        **pooled_metrics,
        "mean_subject_accuracy": float(
            subject_results_df["accuracy"].mean()
        ),
        "subject_accuracy_standard_deviation": float(
            subject_results_df["accuracy"].std(ddof=1)
        ),
        "mean_selected_threshold": float(
            subject_results_df["selected_threshold"].mean()
        ),
        "standard_deviation_selected_threshold": float(
            subject_results_df["selected_threshold"].std(ddof=1)
        ),
        "current_run_wall_minutes": (
            time.perf_counter() - wall_start
        ) / 60.0,
    }

    summary_df = pd.DataFrame([summary])
    atomic_write_csv(summary_df, summary_path)

    atomic_write_json(
        {
            "completed": True,
            "completed_subjects": 223,
            "total_subjects": 223,
            "completed_prediction_rows": len(y),
            "total_prediction_rows": len(y),
            "prediction_path": prediction_path,
            "subject_path": subject_path,
            "summary_path": summary_path,
        },
        progress_path,
    )

    print("\n" + "=" * 76)
    print("V9 CROSS-INDIVIDUAL LOSO COMPLETED")
    print("=" * 76)
    print("Experiment:", experiment_name)
    print("Feature matrix:", X.shape)
    print("Mean selected threshold:", summary["mean_selected_threshold"])

    display(
        summary_df[[
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1_score",
            "roc_auc",
            "average_precision_pr_auc",
            "matthews_correlation_coefficient",
            "cohen_kappa",
        ]].T.rename(columns={0: "value"})
    )

    return {
        "experiment_name": experiment_name,
        "predictions": predictions_df,
        "subject_results": subject_results_df,
        "pooled_summary": summary_df,
        "prediction_path": prediction_path,
        "subject_path": subject_path,
        "summary_path": summary_path,
        "progress_path": progress_path,
        "configuration_path": configuration_path,
    }


print("V9 cross-individual evaluator is ready.")


V9 cross-individual evaluator is ready.


In [ ]:
# ============================================================
# CELL 6 — V8 strict inductive baseline
# ============================================================

v8_strict_cross_result = run_v9_cross_loso(
    X=X_v8_262,
    y=y,
    groups=groups,
    estimator=make_v8_extratrees(),
    model_name="ExtraTrees",
    feature_set_name="V8_Handcrafted262_RawBaselineNormalized",
    experiment_tag="V8_Strict_LOSO_FixedThreshold050",
    nested_threshold=False,
    calibrate=False,
    fixed_threshold=0.50,
    resume=True,
    overwrite=False,
)


V9 LOSO | ExtraTrees | V8_Handcrafted262_RawBaselineNormalized:   0%|          | 0/223 [00:00<?, ?it/s]


V9 CROSS-INDIVIDUAL LOSO COMPLETED
Experiment: ExtraTrees__V8_Handcrafted262_RawBaselineNormalized__V8_Strict_LOSO_FixedThreshold050
Feature matrix: (6275, 262)
Mean selected threshold: 0.5


,value
accuracy,0.640159
balanced_accuracy,0.643851
precision,0.709740
recall,0.615775
specificity,0.671927
f1_score,0.659427
roc_auc,0.687958
average_precision_pr_auc,0.736200
matthews_correlation_coefficient,0.285252
cohen_kappa,0.282049


In [ ]:
# ============================================================
# CELL 7 — V9-A
# ============================================================

v9a_cross_result = run_v9_cross_loso(
    X=X_v8_262,
    y=y,
    groups=groups,
    estimator=make_v8_extratrees(),
    model_name="ExtraTrees",
    feature_set_name="V9A_Handcrafted262",
    experiment_tag="V9A_Sigmoid_NestedThreshold_LOSO",
    nested_threshold=True,
    calibrate=True,
    resume=True,
    overwrite=False,
)


V9 LOSO | ExtraTrees | V9A_Handcrafted262:   0%|          | 0/223 [00:00<?, ?it/s]


V9 CROSS-INDIVIDUAL LOSO COMPLETED
Experiment: ExtraTrees__V9A_Handcrafted262__V9A_Sigmoid_NestedThreshold_LOSO
Feature matrix: (6275, 262)
Mean selected threshold: 0.5987443946188338


,value
accuracy,0.632669
balanced_accuracy,0.645930
precision,0.737143
recall,0.545070
specificity,0.746789
f1_score,0.626721
roc_auc,0.685664
average_precision_pr_auc,0.733649
matthews_correlation_coefficient,0.293265
cohen_kappa,0.280784


## V9-B compact spatial features

V9-B adds 52 interpretable spatial features:

- 15 regional temporal means
- 15 regional N2/P2 morphology features
- 12 left-right difference features
- 6 anterior-posterior gradient features
- 4 global-field-power features

These features are computed independently for each trial from the same-trial prestimulus-normalized response. No labels are used during feature extraction.

In [ ]:
# ============================================================
# CELL 8 — Extract 52 compact spatial features
# ============================================================

V9B_SPATIAL_PATH = os.path.join(
    V9_FEATURE_DIR,
    "X_v9b_compact_spatial_52.npy"
)

V9B_SPATIAL_NAMES_PATH = os.path.join(
    V9_FEATURE_DIR,
    "v9b_compact_spatial_feature_names_52.csv"
)

CHANNEL_INDEX = {
    channel: index
    for index, channel in enumerate(CHANNELS)
}

REGIONS = {
    "frontocentral": ["Fz", "FC1", "FC2"],
    "central": ["C3", "C1", "Cz", "C2", "C4"],
    "centroparietal": ["CP1", "CPz", "CP2", "Pz"],
}

WINDOWS = {
    "early_0_200": (0, 200),
    "n2_180_350": (180, 350),
    "p2_300_600": (300, 600),
    "late_600_1000": (600, 1000),
    "full_0_2000": (0, 2000),
}

LEFT_RIGHT_PAIRS = {
    "FC1_minus_FC2": ("FC1", "FC2"),
    "C1_minus_C2": ("C1", "C2"),
    "C3_minus_C4": ("C3", "C4"),
    "CP1_minus_CP2": ("CP1", "CP2"),
}


def compact_spatial_feature_names():
    names = []

    for region_name in REGIONS:
        for window_name in WINDOWS:
            names.append(
                f"SPATIAL__{region_name}__{window_name}__mean"
            )

    for region_name in REGIONS:
        names.extend([
            f"SPATIAL__{region_name}__n2_amplitude",
            f"SPATIAL__{region_name}__n2_latency_ms",
            f"SPATIAL__{region_name}__p2_amplitude",
            f"SPATIAL__{region_name}__p2_latency_ms",
            f"SPATIAL__{region_name}__n2p2_peak_to_peak",
        ])

    for pair_name in LEFT_RIGHT_PAIRS:
        names.extend([
            f"SPATIAL__{pair_name}__n2_mean_difference",
            f"SPATIAL__{pair_name}__p2_mean_difference",
            f"SPATIAL__{pair_name}__n2p2_difference",
        ])

    for gradient_name in [
        "central_minus_frontocentral",
        "centroparietal_minus_central",
    ]:
        for window_name in [
            "n2_180_350",
            "p2_300_600",
            "full_0_2000",
        ]:
            names.append(
                f"SPATIAL__{gradient_name}__{window_name}__mean"
            )

    names.extend([
        "SPATIAL__gfp__n2_mean",
        "SPATIAL__gfp__n2_max",
        "SPATIAL__gfp__p2_mean",
        "SPATIAL__gfp__p2_max",
    ])

    return names


spatial_feature_names = compact_spatial_feature_names()
assert len(spatial_feature_names) == 52


def extract_compact_spatial_batch(
    baseline_batch,
    response_batch,
    minimum_sd=1e-8,
):
    baseline_batch = np.asarray(
        baseline_batch,
        dtype=np.float64
    )

    response_batch = np.asarray(
        response_batch,
        dtype=np.float64
    )

    baseline_mean = baseline_batch.mean(
        axis=2,
        keepdims=True
    )

    baseline_sd = baseline_batch.std(
        axis=2,
        keepdims=True
    )

    baseline_sd = np.maximum(
        baseline_sd,
        minimum_sd
    )

    normalized_response = (
        response_batch - baseline_mean
    ) / baseline_sd

    region_signals = {}

    for region_name, region_channels in REGIONS.items():
        indices = [
            CHANNEL_INDEX[channel]
            for channel in region_channels
        ]

        region_signals[region_name] = (
            normalized_response[:, indices, :].mean(axis=1)
        )

    feature_columns = []

    for region_name in REGIONS:
        signal = region_signals[region_name]

        for _, (start, stop) in WINDOWS.items():
            feature_columns.append(
                signal[:, start:stop].mean(axis=1)
            )

    for region_name in REGIONS:
        signal = region_signals[region_name]

        n2_segment = signal[:, 180:350]
        p2_segment = signal[:, 300:600]

        n2_amplitude = n2_segment.min(axis=1)
        n2_latency = n2_segment.argmin(axis=1) + 180

        p2_amplitude = p2_segment.max(axis=1)
        p2_latency = p2_segment.argmax(axis=1) + 300

        n2p2 = p2_amplitude - n2_amplitude

        feature_columns.extend([
            n2_amplitude,
            n2_latency.astype(float),
            p2_amplitude,
            p2_latency.astype(float),
            n2p2,
        ])

    for _, (left_channel, right_channel) in LEFT_RIGHT_PAIRS.items():
        left_signal = normalized_response[
            :, CHANNEL_INDEX[left_channel], :
        ]

        right_signal = normalized_response[
            :, CHANNEL_INDEX[right_channel], :
        ]

        n2_mean_difference = (
            left_signal[:, 180:350].mean(axis=1)
            - right_signal[:, 180:350].mean(axis=1)
        )

        p2_mean_difference = (
            left_signal[:, 300:600].mean(axis=1)
            - right_signal[:, 300:600].mean(axis=1)
        )

        left_n2p2 = (
            left_signal[:, 300:600].max(axis=1)
            - left_signal[:, 180:350].min(axis=1)
        )

        right_n2p2 = (
            right_signal[:, 300:600].max(axis=1)
            - right_signal[:, 180:350].min(axis=1)
        )

        feature_columns.extend([
            n2_mean_difference,
            p2_mean_difference,
            left_n2p2 - right_n2p2,
        ])

    gradient_pairs = [
        (
            region_signals["central"],
            region_signals["frontocentral"],
        ),
        (
            region_signals["centroparietal"],
            region_signals["central"],
        ),
    ]

    gradient_windows = [
        (180, 350),
        (300, 600),
        (0, 2000),
    ]

    for posterior_signal, anterior_signal in gradient_pairs:
        difference_signal = posterior_signal - anterior_signal

        for start, stop in gradient_windows:
            feature_columns.append(
                difference_signal[:, start:stop].mean(axis=1)
            )

    global_field_power = normalized_response.std(axis=1)

    feature_columns.extend([
        global_field_power[:, 180:350].mean(axis=1),
        global_field_power[:, 180:350].max(axis=1),
        global_field_power[:, 300:600].mean(axis=1),
        global_field_power[:, 300:600].max(axis=1),
    ])

    feature_matrix = np.column_stack(feature_columns)

    if feature_matrix.shape[1] != 52:
        raise RuntimeError(
            f"Expected 52 spatial features, "
            f"found {feature_matrix.shape[1]}."
        )

    return feature_matrix.astype(np.float32)


if os.path.exists(V9B_SPATIAL_PATH):
    X_v9b_spatial = np.load(
        V9B_SPATIAL_PATH,
        mmap_mode="r"
    )

    if X_v9b_spatial.shape != (6275, 52):
        raise ValueError(
            "Existing V9-B spatial file has an unexpected shape."
        )

    print("Loaded existing V9-B spatial features.")

else:
    X_v9b_spatial_writer = np.lib.format.open_memmap(
        V9B_SPATIAL_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(6275, 52),
    )

    batch_size = 64

    for start in tqdm(
        range(0, 6275, batch_size),
        desc="Extracting V9-B compact spatial features",
    ):
        stop = min(start + batch_size, 6275)

        X_v9b_spatial_writer[start:stop] = (
            extract_compact_spatial_batch(
                X_baseline_eeg[start:stop],
                X_response_eeg[start:stop],
            )
        )

    X_v9b_spatial_writer.flush()
    del X_v9b_spatial_writer

    pd.DataFrame({
        "feature_index": np.arange(52),
        "feature_name": spatial_feature_names,
    }).to_csv(
        V9B_SPATIAL_NAMES_PATH,
        index=False,
    )

    X_v9b_spatial = np.load(
        V9B_SPATIAL_PATH,
        mmap_mode="r"
    )

assert X_v9b_spatial.shape == (6275, 52)

for start in range(0, 6275, 512):
    stop = min(start + 512, 6275)
    if not np.isfinite(
        np.asarray(X_v9b_spatial[start:stop])
    ).all():
        raise ValueError("V9-B contains non-finite values.")

print("V9-B compact spatial matrix:", X_v9b_spatial.shape)


Extracting V9-B compact spatial features:   0%|          | 0/99 [00:00<?, ?it/s]

V9-B compact spatial matrix: (6275, 52)


In [ ]:
# ============================================================
# CELL 9 — Combine V8 temporal features + V9-B compact spatial
# ============================================================

V9B_COMBINED_PATH = os.path.join(
    V9_FEATURE_DIR,
    "X_v9b_temporal262_compactspatial52_314.npy"
)

V9B_COMBINED_NAMES_PATH = os.path.join(
    V9_FEATURE_DIR,
    "v9b_feature_names_314.csv"
)


def combine_feature_matrices(
    first_matrix,
    second_matrix,
    output_path,
    dtype=np.float32,
    batch_size=512,
):
    expected_shape = (
        first_matrix.shape[0],
        first_matrix.shape[1] + second_matrix.shape[1],
    )

    if os.path.exists(output_path):
        existing = np.load(
            output_path,
            mmap_mode="r"
        )

        if existing.shape != expected_shape:
            raise ValueError(
                f"Existing combined file has shape "
                f"{existing.shape}; expected {expected_shape}."
            )

        return existing

    writer = np.lib.format.open_memmap(
        output_path,
        mode="w+",
        dtype=dtype,
        shape=expected_shape,
    )

    for start in tqdm(
        range(
            0,
            expected_shape[0],
            batch_size
        ),
        desc=(
            f"Combining features into "
            f"{expected_shape[1]} columns"
        ),
    ):
        stop = min(
            start + batch_size,
            expected_shape[0]
        )

        writer[
            start:stop,
            :first_matrix.shape[1]
        ] = np.asarray(
            first_matrix[start:stop],
            dtype=dtype,
        )

        writer[
            start:stop,
            first_matrix.shape[1]:
        ] = np.asarray(
            second_matrix[start:stop],
            dtype=dtype,
        )

    writer.flush()
    del writer

    return np.load(
        output_path,
        mmap_mode="r"
    )


X_v9b_314 = combine_feature_matrices(
    first_matrix=X_v8_262,
    second_matrix=X_v9b_spatial,
    output_path=V9B_COMBINED_PATH,
)


v9b_feature_names = (
    feature_names_262
    + spatial_feature_names
)


assert X_v8_262.shape == (6275, 262)
assert X_v9b_spatial.shape == (6275, 52)
assert X_v9b_314.shape == (6275, 314)

assert len(feature_names_262) == 262
assert len(spatial_feature_names) == 52
assert len(v9b_feature_names) == 314


pd.DataFrame({
    "feature_index": np.arange(314),
    "feature_name": v9b_feature_names,
}).to_csv(
    V9B_COMBINED_NAMES_PATH,
    index=False,
)


print("=" * 70)
print("V9-B FEATURE FUSION COMPLETED")
print("=" * 70)

print(
    "V8 handcrafted features:",
    X_v8_262.shape
)

print(
    "Compact spatial features:",
    X_v9b_spatial.shape
)

print(
    "Combined V9-B matrix:",
    X_v9b_314.shape
)

print(
    "Saved combined matrix:",
    V9B_COMBINED_PATH
)

print(
    "Saved feature names:",
    V9B_COMBINED_NAMES_PATH
)

V9-B FEATURE FUSION COMPLETED
V8 handcrafted features: (6275, 262)
Compact spatial features: (6275, 52)
Combined V9-B matrix: (6275, 314)
Saved combined matrix: /content/drive/MyDrive/EXP8_CLASSIFICATION_FROM_SCRATCH/V9_MAIN_PIPELINE_12CH_SPATIAL/features/X_v9b_temporal262_compactspatial52_314.npy
Saved feature names: /content/drive/MyDrive/EXP8_CLASSIFICATION_FROM_SCRATCH/V9_MAIN_PIPELINE_12CH_SPATIAL/features/v9b_feature_names_314.csv


In [ ]:
# ============================================================
# CELL 10 — Run V9-B cross-individual LOSO
# ============================================================

v9b_cross_result = run_v9_cross_loso(
    X=X_v9b_314,
    y=y,
    groups=groups,
    estimator=make_v8_extratrees(),
    model_name="ExtraTrees",
    feature_set_name="V9B_Temporal262_CompactSpatial52",
    experiment_tag="V9B_Sigmoid_NestedThreshold_LOSO",
    nested_threshold=True,
    calibrate=True,
    resume=True,
    overwrite=False,
)


V9 LOSO | ExtraTrees | V9B_Temporal262_CompactSpatial52:   0%|          | 0/223 [00:00<?, ?it/s]


V9 CROSS-INDIVIDUAL LOSO COMPLETED
Experiment: ExtraTrees__V9B_Temporal262_CompactSpatial52__V9B_Sigmoid_NestedThreshold_LOSO
Feature matrix: (6275, 314)
Mean selected threshold: 0.599775784753363


,value
accuracy,0.628845
balanced_accuracy,0.641739
precision,0.731338
recall,0.543662
specificity,0.739817
f1_score,0.623687
roc_auc,0.683628
average_precision_pr_auc,0.730993
matthews_correlation_coefficient,0.284633
cohen_kappa,0.272878


## V9-C baseline-whitened spatial covariance

V9-C adds two 78-dimensional symmetric log-covariance representations:

- N2 interval: 180–350 ms
- P2 interval: 300–600 ms

Total covariance features: `78 × 2 = 156`.

The covariance features are calculated independently for each trial and use that trial's own prestimulus covariance for whitening. No labels or held-out-participant information are used during feature extraction.

In [ ]:
# ============================================================
# CELL 11 — Extract 156 baseline-whitened log-covariance features
# ============================================================

V9C_COVARIANCE_PATH = os.path.join(
    V9_FEATURE_DIR,
    "X_v9c_baseline_whitened_logcov_156.npy"
)

V9C_COVARIANCE_NAMES_PATH = os.path.join(
    V9_FEATURE_DIR,
    "v9c_baseline_whitened_logcov_feature_names_156.csv"
)

COVARIANCE_WINDOWS = {
    "N2_180_350": (180, 350),
    "P2_300_600": (300, 600),
}

COVARIANCE_SHRINKAGE = 0.10
SPD_EPSILON = 1e-6

UPPER_TRIANGLE = np.triu_indices(12)
assert len(UPPER_TRIANGLE[0]) == 78


def regularized_covariance(
    signal,
    shrinkage=COVARIANCE_SHRINKAGE,
    epsilon=SPD_EPSILON,
):
    signal = np.asarray(signal, dtype=np.float64)
    signal = signal - signal.mean(axis=1, keepdims=True)

    denominator = max(1, signal.shape[1] - 1)
    covariance = signal @ signal.T / denominator

    average_variance = np.trace(covariance) / covariance.shape[0]

    covariance = (
        (1.0 - shrinkage) * covariance
        + shrinkage
        * average_variance
        * np.eye(covariance.shape[0])
    )

    covariance = (
        0.5 * (covariance + covariance.T)
        + epsilon * np.eye(covariance.shape[0])
    )

    return covariance


def symmetric_inverse_square_root(matrix):
    eigenvalues, eigenvectors = np.linalg.eigh(
        0.5 * (matrix + matrix.T)
    )

    eigenvalues = np.maximum(
        eigenvalues,
        SPD_EPSILON
    )

    return (
        eigenvectors
        @ np.diag(eigenvalues ** -0.5)
        @ eigenvectors.T
    )


def symmetric_matrix_log(matrix):
    eigenvalues, eigenvectors = np.linalg.eigh(
        0.5 * (matrix + matrix.T)
    )

    eigenvalues = np.maximum(
        eigenvalues,
        SPD_EPSILON
    )

    return (
        eigenvectors
        @ np.diag(np.log(eigenvalues))
        @ eigenvectors.T
    )


def symmetric_matrix_to_vector(matrix):
    vector = matrix[UPPER_TRIANGLE].astype(float)

    off_diagonal = (
        UPPER_TRIANGLE[0]
        != UPPER_TRIANGLE[1]
    )

    vector[off_diagonal] *= np.sqrt(2.0)
    return vector


def covariance_feature_names():
    names = []

    for window_name in COVARIANCE_WINDOWS:
        for row_index, column_index in zip(
            UPPER_TRIANGLE[0],
            UPPER_TRIANGLE[1],
        ):
            names.append(
                f"LOGCOV__{window_name}__"
                f"{CHANNELS[row_index]}_{CHANNELS[column_index]}"
            )

    return names


covariance_feature_names_156 = covariance_feature_names()
assert len(covariance_feature_names_156) == 156


def extract_one_trial_logcov(
    baseline_trial,
    response_trial,
):
    baseline_trial = np.asarray(
        baseline_trial,
        dtype=np.float64
    )

    response_trial = np.asarray(
        response_trial,
        dtype=np.float64
    )

    baseline_mean = baseline_trial.mean(
        axis=1,
        keepdims=True
    )

    baseline_sd = baseline_trial.std(
        axis=1,
        keepdims=True
    )

    baseline_sd = np.maximum(
        baseline_sd,
        1e-8
    )

    normalized_baseline = (
        baseline_trial - baseline_mean
    ) / baseline_sd

    normalized_response = (
        response_trial - baseline_mean
    ) / baseline_sd

    baseline_covariance = regularized_covariance(
        normalized_baseline
    )

    baseline_inverse_sqrt = (
        symmetric_inverse_square_root(
            baseline_covariance
        )
    )

    features = []

    for start, stop in COVARIANCE_WINDOWS.values():
        response_covariance = regularized_covariance(
            normalized_response[:, start:stop]
        )

        whitened_covariance = (
            baseline_inverse_sqrt
            @ response_covariance
            @ baseline_inverse_sqrt
        )

        whitened_covariance = 0.5 * (
            whitened_covariance
            + whitened_covariance.T
        )

        log_covariance = symmetric_matrix_log(
            whitened_covariance
        )

        features.append(
            symmetric_matrix_to_vector(
                log_covariance
            )
        )

    return np.concatenate(features).astype(np.float32)


if os.path.exists(V9C_COVARIANCE_PATH):
    X_v9c_covariance = np.load(
        V9C_COVARIANCE_PATH,
        mmap_mode="r"
    )

    if X_v9c_covariance.shape != (6275, 156):
        raise ValueError(
            "Existing V9-C covariance file has an unexpected shape."
        )

    print("Loaded existing V9-C covariance features.")

else:
    X_v9c_covariance_writer = np.lib.format.open_memmap(
        V9C_COVARIANCE_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(6275, 156),
    )

    for epoch_index in tqdm(
        range(6275),
        desc="Extracting V9-C log-covariance features",
    ):
        X_v9c_covariance_writer[epoch_index] = (
            extract_one_trial_logcov(
                X_baseline_eeg[epoch_index],
                X_response_eeg[epoch_index],
            )
        )

        if (
            epoch_index > 0
            and epoch_index % 250 == 0
        ):
            X_v9c_covariance_writer.flush()

    X_v9c_covariance_writer.flush()
    del X_v9c_covariance_writer

    pd.DataFrame({
        "feature_index": np.arange(156),
        "feature_name": covariance_feature_names_156,
    }).to_csv(
        V9C_COVARIANCE_NAMES_PATH,
        index=False,
    )

    X_v9c_covariance = np.load(
        V9C_COVARIANCE_PATH,
        mmap_mode="r"
    )

assert X_v9c_covariance.shape == (6275, 156)

for start in range(0, 6275, 512):
    stop = min(start + 512, 6275)

    if not np.isfinite(
        np.asarray(X_v9c_covariance[start:stop])
    ).all():
        raise ValueError("V9-C contains non-finite values.")

print("V9-C covariance matrix:", X_v9c_covariance.shape)


Extracting V9-C log-covariance features:   0%|          | 0/6275 [00:00<?, ?it/s]

V9-C covariance matrix: (6275, 156)


In [ ]:
# ============================================================
# CELL 12 — Combine V8 temporal + V9-C covariance
# ============================================================

V9C_COMBINED_PATH = os.path.join(
    V9_FEATURE_DIR,
    "X_v9c_temporal262_logcov156_418.npy"
)

V9C_COMBINED_NAMES_PATH = os.path.join(
    V9_FEATURE_DIR,
    "v9c_feature_names_418.csv"
)

X_v9c_418 = combine_feature_matrices(
    X_v8_262,
    X_v9c_covariance,
    V9C_COMBINED_PATH,
)

v9c_feature_names = (
    feature_names_262
    + covariance_feature_names_156
)

assert len(v9c_feature_names) == 418
assert X_v9c_418.shape == (6275, 418)

pd.DataFrame({
    "feature_index": np.arange(418),
    "feature_name": v9c_feature_names,
}).to_csv(
    V9C_COMBINED_NAMES_PATH,
    index=False,
)

print("V9-C combined matrix:", X_v9c_418.shape)


Combining features into 418 columns:   0%|          | 0/13 [00:00<?, ?it/s]

V9-C combined matrix: (6275, 418)


In [ ]:
# ============================================================
# CELL 13 — Run V9-C cross-individual LOSO
# ============================================================

v9c_cross_result = run_v9_cross_loso(
    X=X_v9c_418,
    y=y,
    groups=groups,
    estimator=make_v8_extratrees(),
    model_name="ExtraTrees",
    feature_set_name="V9C_Temporal262_LogCovariance156",
    experiment_tag="V9C_Sigmoid_NestedThreshold_LOSO",
    nested_threshold=True,
    calibrate=True,
    resume=True,
    overwrite=False,
)


In [ ]:
# ============================================================
# CELL 14 — Compare V8 strict, V9-A, V9-B and V9-C
# ============================================================

cross_comparison_df = pd.concat(
    [
        v8_strict_cross_result["pooled_summary"],
        v9a_cross_result["pooled_summary"],
        v9b_cross_result["pooled_summary"],
        v9c_cross_result["pooled_summary"],
    ],
    ignore_index=True,
)

cross_comparison_columns = [
    "experiment",
    "n_features",
    "accuracy",
    "balanced_accuracy",
    "f1_score",
    "precision",
    "recall",
    "specificity",
    "roc_auc",
    "average_precision_pr_auc",
    "matthews_correlation_coefficient",
    "mean_selected_threshold",
]

display(
    cross_comparison_df[
        cross_comparison_columns
    ]
)

CROSS_COMPARISON_PATH = os.path.join(
    V9_CROSS_RESULTS_DIR,
    "V9_cross_individual_model_comparison.csv"
)

cross_comparison_df[
    cross_comparison_columns
].to_csv(
    CROSS_COMPARISON_PATH,
    index=False,
)

print("Saved:", CROSS_COMPARISON_PATH)


# Optional within-individual branch

The following cells use the same fixed Extra Trees configuration. They first create raw within-individual leave-one-trial-out probabilities. A second stage learns calibration and threshold settings from **other participants only**, then applies them to the target participant.

In [ ]:
# ============================================================
# CELL 15 — Generic within-individual raw-score evaluator
# ============================================================

def eligible_within_subjects(y, groups):
    balance = (
        pd.DataFrame({
            "subject_id": np.asarray(groups).astype(str),
            "label": np.asarray(y, dtype=np.int8),
        })
        .groupby(["subject_id", "label"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=[0, 1], fill_value=0)
    )

    return np.sort(
        balance.index[
            (balance[0] >= 2)
            & (balance[1] >= 2)
        ].astype(str).to_numpy()
    )


def run_v9_within_raw_scores(
    X,
    y,
    groups,
    estimator,
    model_name,
    feature_set_name,
    experiment_tag,
    resume=True,
    overwrite=False,
):
    y = np.asarray(y, dtype=np.int8).ravel()
    groups = np.asarray(groups).astype(str).ravel()

    subjects = eligible_within_subjects(y, groups)

    if len(subjects) != 192:
        raise ValueError(
            f"Expected 192 eligible participants, found {len(subjects)}."
        )

    strict_indices = np.flatnonzero(
        np.isin(groups, subjects)
    )

    if len(strict_indices) != 5408:
        raise ValueError(
            f"Expected 5408 strict trials, found {len(strict_indices)}."
        )

    experiment_name = (
        f"{safe_name(model_name)}__"
        f"{safe_name(feature_set_name)}__"
        f"{safe_name(experiment_tag)}"
    )

    prediction_path = os.path.join(
        V9_WITHIN_RESULTS_DIR,
        experiment_name + "__raw_trial_predictions.csv"
    )

    summary_path = os.path.join(
        V9_WITHIN_RESULTS_DIR,
        experiment_name + "__raw_pooled_summary.csv"
    )

    progress_path = os.path.join(
        V9_WITHIN_RESULTS_DIR,
        experiment_name + "__progress.json"
    )

    if overwrite:
        for path in [
            prediction_path,
            summary_path,
            progress_path,
        ]:
            if os.path.exists(path):
                os.remove(path)

    if resume and os.path.exists(prediction_path):
        predictions_df = pd.read_csv(prediction_path)
        predictions_df["test_subject"] = (
            predictions_df["test_subject"].astype(str)
        )
        predictions_df = (
            predictions_df
            .sort_values("original_row_index")
            .drop_duplicates("original_row_index", keep="last")
            .reset_index(drop=True)
        )
        completed_subjects = set(
            predictions_df["test_subject"].unique()
        )
    else:
        predictions_df = pd.DataFrame()
        completed_subjects = set()

    wall_start = time.perf_counter()

    for subject_id in tqdm(
        subjects,
        desc=f"V9 within LOOCV | {feature_set_name}",
    ):
        subject_id = str(subject_id)

        if subject_id in completed_subjects:
            continue

        subject_indices = np.flatnonzero(
            groups == subject_id
        )

        subject_rows = []

        for test_index in subject_indices:
            train_indices = subject_indices[
                subject_indices != test_index
            ]

            y_train = y[train_indices]

            if len(np.unique(y_train)) != 2:
                raise RuntimeError(
                    f"{subject_id}: training fold lacks one class."
                )

            fold_model = clone(estimator)
            fold_model.fit(
                X[train_indices],
                y_train,
            )

            raw_score = positive_probability(
                fold_model,
                X[[test_index]],
            )[0]

            subject_rows.append({
                "experiment": experiment_name,
                "model": model_name,
                "feature_set": feature_set_name,
                "test_subject": subject_id,
                "original_row_index": int(test_index),
                "y_true": int(y[test_index]),
                "raw_probability": float(raw_score),
                "y_score": float(raw_score),
                "y_pred": int(raw_score >= 0.50),
                "selected_threshold": 0.50,
            })

        predictions_df = pd.concat(
            [
                predictions_df,
                pd.DataFrame(subject_rows),
            ],
            ignore_index=True,
        )

        predictions_df = (
            predictions_df
            .sort_values("original_row_index")
            .drop_duplicates("original_row_index", keep="last")
            .reset_index(drop=True)
        )

        atomic_write_csv(
            predictions_df,
            prediction_path
        )

        completed_subjects.add(subject_id)

        atomic_write_json(
            {
                "completed": False,
                "completed_subjects": len(completed_subjects),
                "total_subjects": 192,
                "completed_prediction_rows": len(predictions_df),
                "total_prediction_rows": 5408,
                "last_completed_subject": subject_id,
            },
            progress_path,
        )

    predictions_df = pd.read_csv(prediction_path)
    predictions_df["test_subject"] = (
        predictions_df["test_subject"].astype(str)
    )
    predictions_df = (
        predictions_df
        .sort_values("original_row_index")
        .drop_duplicates("original_row_index", keep="last")
        .reset_index(drop=True)
    )

    if len(predictions_df) != 5408:
        raise RuntimeError(
            f"Within evaluation incomplete: "
            f"{len(predictions_df)} of 5408."
        )

    pooled = binary_metrics(
        predictions_df["y_true"],
        predictions_df["y_pred"],
        predictions_df["y_score"],
    )

    pooled.update({
        "experiment": experiment_name,
        "model": model_name,
        "feature_set": feature_set_name,
        "n_features": int(X.shape[1]),
        "n_subjects": 192,
        "n_trials": 5408,
        "threshold_method": "fixed_0.50",
        "current_run_wall_minutes": (
            time.perf_counter() - wall_start
        ) / 60.0,
    })

    summary_df = pd.DataFrame([pooled])
    atomic_write_csv(summary_df, summary_path)

    atomic_write_json(
        {
            "completed": True,
            "completed_subjects": 192,
            "total_subjects": 192,
            "completed_prediction_rows": 5408,
            "total_prediction_rows": 5408,
        },
        progress_path,
    )

    return {
        "predictions": predictions_df,
        "pooled_summary": summary_df,
        "prediction_path": prediction_path,
        "summary_path": summary_path,
    }


print("Generic V9 within-individual evaluator is ready.")


In [ ]:
# ============================================================
# CELL 16 — Leakage-free threshold transfer for within scores
# ============================================================

def calibrate_within_scores_from_other_subjects(
    raw_predictions,
    output_name,
    calibrate=True,
    threshold_grid=THRESHOLD_GRID,
):
    predictions_df = raw_predictions.copy()
    predictions_df["test_subject"] = (
        predictions_df["test_subject"].astype(str)
    )

    adjusted_rows = []

    all_subjects = np.sort(
        predictions_df["test_subject"].unique()
    )

    for target_subject in tqdm(
        all_subjects,
        desc=f"Threshold transfer | {output_name}",
    ):
        training_df = predictions_df[
            predictions_df["test_subject"]
            != target_subject
        ].copy()

        target_df = predictions_df[
            predictions_df["test_subject"]
            == target_subject
        ].copy()

        training_raw_scores = training_df[
            "raw_probability"
        ].to_numpy(float)

        target_raw_scores = target_df[
            "raw_probability"
        ].to_numpy(float)

        if calibrate:
            calibrator = fit_sigmoid_calibrator(
                training_raw_scores,
                training_df["y_true"],
            )

            training_scores = apply_sigmoid_calibrator(
                calibrator,
                training_raw_scores,
            )

            target_scores = apply_sigmoid_calibrator(
                calibrator,
                target_raw_scores,
            )
        else:
            training_scores = training_raw_scores
            target_scores = target_raw_scores

        (
            selected_threshold,
            selected_row,
            selection_reason,
            _,
        ) = choose_threshold(
            training_df["y_true"],
            training_scores,
            threshold_grid=threshold_grid,
        )

        target_df["y_score"] = target_scores
        target_df["selected_threshold"] = selected_threshold
        target_df["y_pred"] = (
            target_scores >= selected_threshold
        ).astype(np.int8)

        target_df[
            "threshold_selection_reason"
        ] = selection_reason

        target_df["threshold_training_precision"] = (
            selected_row["precision"]
        )

        target_df["threshold_training_recall"] = (
            selected_row["recall"]
        )

        target_df["threshold_training_specificity"] = (
            selected_row["specificity"]
        )

        adjusted_rows.append(target_df)

    adjusted_df = (
        pd.concat(adjusted_rows, ignore_index=True)
        .sort_values("original_row_index")
        .reset_index(drop=True)
    )

    pooled = binary_metrics(
        adjusted_df["y_true"],
        adjusted_df["y_pred"],
        adjusted_df["y_score"],
    )

    pooled.update({
        "experiment": output_name,
        "n_subjects": int(
            adjusted_df["test_subject"].nunique()
        ),
        "n_trials": int(len(adjusted_df)),
        "threshold_method": (
            "leave-one-subject-out transferred "
            "sigmoid calibration and threshold"
        ),
        "mean_selected_threshold": float(
            adjusted_df
            .groupby("test_subject")[
                "selected_threshold"
            ]
            .first()
            .mean()
        ),
    })

    output_prediction_path = os.path.join(
        V9_WITHIN_RESULTS_DIR,
        safe_name(output_name)
        + "__adjusted_predictions.csv"
    )

    output_summary_path = os.path.join(
        V9_WITHIN_RESULTS_DIR,
        safe_name(output_name)
        + "__adjusted_summary.csv"
    )

    atomic_write_csv(
        adjusted_df,
        output_prediction_path
    )

    pooled_df = pd.DataFrame([pooled])
    atomic_write_csv(
        pooled_df,
        output_summary_path
    )

    display(
        pooled_df[[
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1_score",
            "roc_auc",
            "average_precision_pr_auc",
            "mean_selected_threshold",
        ]].T.rename(columns={0: "value"})
    )

    return {
        "predictions": adjusted_df,
        "pooled_summary": pooled_df,
        "prediction_path": output_prediction_path,
        "summary_path": output_summary_path,
    }


print("Within-individual threshold-transfer helper is ready.")


In [ ]:
# ============================================================
# CELL 17 — Existing V8 within-individual predictions
# ============================================================

V8_WITHIN_PREDICTION_PATH = os.path.join(
    V8_DIR,
    "V8_WITHIN_INDIVIDUAL_LOOCV_RESULTS_STRICT192",
    (
        "ExtraTrees__Handcrafted_262_noPCA__"
        "v8_full0_2000ms_no_pca_strict192"
        "__trial_predictions.csv"
    ),
)

if not os.path.exists(V8_WITHIN_PREDICTION_PATH):
    raise FileNotFoundError(
        "The completed V8 within-individual prediction file "
        "was not found:\n"
        + V8_WITHIN_PREDICTION_PATH
    )

v8_within_raw_predictions = pd.read_csv(
    V8_WITHIN_PREDICTION_PATH
)

required_columns = {
    "test_subject",
    "original_row_index",
    "y_true",
    "y_score",
}

missing_columns = required_columns.difference(
    v8_within_raw_predictions.columns
)

if missing_columns:
    raise ValueError(
        f"V8 prediction file is missing: "
        f"{sorted(missing_columns)}"
    )

v8_within_raw_predictions[
    "raw_probability"
] = v8_within_raw_predictions["y_score"]

v9a_within_adjusted_result = (
    calibrate_within_scores_from_other_subjects(
        raw_predictions=v8_within_raw_predictions,
        output_name=(
            "V9A_Within_Handcrafted262_"
            "TransferredCalibrationThreshold"
        ),
        calibrate=True,
    )
)


In [ ]:
# ============================================================
# CELL 18 — V9-B within-individual raw scores and adjustment
# ============================================================

v9b_within_raw_result = run_v9_within_raw_scores(
    X=X_v9b_314,
    y=y,
    groups=groups,
    estimator=make_v8_extratrees(),
    model_name="ExtraTrees",
    feature_set_name="V9B_Temporal262_CompactSpatial52",
    experiment_tag="V9B_Within_RawScores",
    resume=True,
    overwrite=False,
)

v9b_within_adjusted_result = (
    calibrate_within_scores_from_other_subjects(
        raw_predictions=v9b_within_raw_result["predictions"],
        output_name=(
            "V9B_Within_Temporal262_CompactSpatial52_"
            "TransferredCalibrationThreshold"
        ),
        calibrate=True,
    )
)


In [ ]:
# ============================================================
# CELL 19 — V9-C within-individual raw scores and adjustment
# ============================================================

v9c_within_raw_result = run_v9_within_raw_scores(
    X=X_v9c_418,
    y=y,
    groups=groups,
    estimator=make_v8_extratrees(),
    model_name="ExtraTrees",
    feature_set_name="V9C_Temporal262_LogCovariance156",
    experiment_tag="V9C_Within_RawScores",
    resume=True,
    overwrite=False,
)

v9c_within_adjusted_result = (
    calibrate_within_scores_from_other_subjects(
        raw_predictions=v9c_within_raw_result["predictions"],
        output_name=(
            "V9C_Within_Temporal262_LogCovariance156_"
            "TransferredCalibrationThreshold"
        ),
        calibrate=True,
    )
)


In [ ]:
# ============================================================
# CELL 20 — Final within-individual comparison
# ============================================================

within_comparison_df = pd.concat(
    [
        v9a_within_adjusted_result["pooled_summary"],
        v9b_within_adjusted_result["pooled_summary"],
        v9c_within_adjusted_result["pooled_summary"],
    ],
    ignore_index=True,
)

within_columns = [
    "experiment",
    "accuracy",
    "balanced_accuracy",
    "f1_score",
    "precision",
    "recall",
    "specificity",
    "roc_auc",
    "average_precision_pr_auc",
    "matthews_correlation_coefficient",
    "mean_selected_threshold",
]

display(
    within_comparison_df[within_columns]
)

WITHIN_COMPARISON_PATH = os.path.join(
    V9_WITHIN_RESULTS_DIR,
    "V9_within_individual_model_comparison.csv"
)

within_comparison_df[
    within_columns
].to_csv(
    WITHIN_COMPARISON_PATH,
    index=False,
)

print("Saved:", WITHIN_COMPARISON_PATH)


## Recommended execution order

Run Cells 1–7 first and inspect V9-A.

Only then run:

- Cells 8–10 for V9-B.
- Cells 11–13 for V9-C.
- Cell 14 for cross-individual comparison.
- Cells 15–20 only when you are ready for the optional within-individual branch.

Do not alter the threshold grid or benchmark constraints after inspecting outer LOSO results. Lock the protocol before treating the final output as the thesis result.